# Arm D (governance) - Mistral-7B (part 1 of 2)

**Run this notebook as-is. Do not edit anything.**

Third family for the governance arm.

## What to do

1. Open it in Kaggle with the **GPU** toggle on, then **Run All**.
2. Wait for the last cell. It ends with a line reading `ALL CELLS COMPLETE`.
   That is the only signal that matters - if the notebook stopped early, read the
   message it printed and re-run it, but see the note below.
3. The last cell leaves one file, `elicit_results_D_mistral_7bpart1.tar.gz`. Click it to
   download, and keep it.
4. Run notebook **12** next.

## About the time budget

This notebook is sized for **one Kaggle session** (5 runs, about
6.5 h of simulation, plus ~0.6 h of setup).

Kaggle stops a GPU session at 12 hours, and **Save & Run All always starts from
a clean session** - it cannot pick up where a previous run stopped. So each
notebook is deliberately sized to finish in one go rather than relying on
resuming. If a run is slower than estimated and the notebook stops early,
re-running it repeats the cells it already finished; that is wasted quota, not
data corruption, and the notebooks are split conservatively to make it unlikely.

In [ ]:
# ============================================================================
# Configuration - already set. Running this notebook as-is is the whole procedure.
# ============================================================================
import io
import json
import os
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

SERVED_MODEL = "mistral-7b"
ARM = "D"
GRID = "configs/grid_arm_d.yaml"
CELLS = 5
CHUNK_START = 0
PER_CELL_HOURS = 1.3
RUNTIME_BUDGET_S = int(12.0 * 3600)
SETUP_ALLOWANCE_S = int(0.6 * 3600)

REPO = "https://github.com/uzairlol/ELICIT-fyp.git"
REPO_DIR = Path("/kaggle/working/elicit")

print(f"arm           : {ARM}")
print(f"served model  : {SERVED_MODEL}")
print(f"grid manifest : {GRID}")
print(f"this notebook : {CELLS} runs of arm {ARM} on {SERVED_MODEL}")
print(f"estimate      : ~{CELLS * PER_CELL_HOURS:.1f} h of simulation")
print(f"hard stop     : {RUNTIME_BUDGET_S / 3600:.1f} h (Kaggle's cap is 12 h)")
print(f"session start : {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}")

In [ ]:
# ============================================================================
# Clone the repo and install its Python dependencies
# ============================================================================
if not REPO_DIR.exists():
    !git clone -q {REPO} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))

print("environment:")
!python --version
!free -h | head -2
!df -h /kaggle | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

!pip install -qq -r requirements.txt
!pip install -qq -e . --no-deps
print("dependencies installed.")

In [ ]:
# ============================================================================
# Install vLLM into an isolated venv
# ============================================================================
# Kaggle's base image ships RAPIDS and protobuf 6.x, and a plain `pip install vllm`
# walks into broken dependency resolution there. An isolated venv keeps vLLM's
# resolver clean and leaves the base environment (which the simulation imports
# from) untouched; the two only talk to each other over HTTP.
VENV = "/kaggle/working/vllm-env"

!apt-get update -y -qq > /dev/null && apt-get install -y -qq pciutils zstd > /dev/null

if not Path(VENV).exists():
    !python -m pip install -q --upgrade pip virtualenv
    !python -m virtualenv {VENV}

!env -u PYTHONPATH -u PYTHONHOME {VENV}/bin/pip install -q --upgrade pip
print("installing vLLM (large CUDA torch wheel, several minutes) ...")
!env -u PYTHONPATH -u PYTHONHOME {VENV}/bin/pip install -q vllm huggingface_hub

!env -u PYTHONPATH -u PYTHONHOME {VENV}/bin/python -c "import vllm, torch; print('  vllm', vllm.__version__, '| cuda', torch.cuda.is_available(), '| gpus', torch.cuda.device_count())"

In [ ]:
# ============================================================================
# Start the vLLM server for this session's model
# ============================================================================
# The flags are held identical across all three families (tensor-parallel 2,
# max-model-len 8192, max-num-seqs 32). Only the checkpoint differs between
# notebooks. If the serving configuration differed between families, "which
# model produced this result" would be unanswerable.
VLLM_LOG = "/kaggle/working/vllm.log"
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

serve_cmd = ["bash", "scripts/serve_arm_model.sh", SERVED_MODEL]
print("starting:", " ".join(serve_cmd), flush=True)

with open(VLLM_LOG, "w", encoding="utf-8") as handle:
    process = subprocess.Popen(
        serve_cmd,
        stdout=handle,
        stderr=subprocess.STDOUT,
        env={**os.environ, "VLLM_BIN": f"{VENV}/bin/vllm"},
    )
print(f"server pid {process.pid}; log -> {VLLM_LOG}")
print("downloading weights and loading the engine, up to ~30 min ...", flush=True)

deadline = time.time() + 45 * 60
ready = False
while time.time() < deadline:
    if process.poll() is not None:
        print("server exited early; log tail:")
        !tail -40 {VLLM_LOG}
        raise SystemExit("vLLM failed to start")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5) as response:
            names = [m.get("id") for m in json.loads(response.read()).get("data", [])]
        if SERVED_MODEL in names:
            ready = True
            break
    except Exception:
        pass
    time.sleep(10)

if not ready:
    !tail -60 {VLLM_LOG}
    raise SystemExit(f"server did not come up as {SERVED_MODEL}")
print("server ready.")
!tail -4 {VLLM_LOG}

In [ ]:
# ============================================================================
# Smoke test - one real request before spending quota
# ============================================================================
# Cheap insurance. If the endpoint or the chat path is broken, find out in
# seconds rather than after the first hour-long run.
body = json.dumps(
    {
        "model": SERVED_MODEL,
        "messages": [{"role": "user", "content": "Reply with exactly: READY"}],
        "max_tokens": 8,
        "temperature": 0.0,
    }
).encode("utf-8")
request = urllib.request.Request(
    "http://127.0.0.1:8000/v1/chat/completions",
    data=body,
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(request, timeout=120) as response:
    reply = json.loads(response.read())
print("server responds:", repr(reply["choices"][0]["message"]["content"].strip()))

In [ ]:
# ============================================================================
# Work out which cells this session is responsible for
# ============================================================================
from cli import main as cli_main


def elicit(*args):
    """Run the CLI in-process and parse its JSON output."""
    buffer = sys.stdout
    captured = io.StringIO()
    sys.stdout = captured
    try:
        code = cli_main(list(args))
    finally:
        sys.stdout = buffer
    text = captured.getvalue()
    try:
        return code, json.loads(text)
    except ValueError:
        return code, {"raw": text}


_code, plan = elicit("grid", "show", "--grid", GRID, "--arm", ARM)

# This notebook owns a chunk of one model's cells for one arm. Cells for the
# other two families belong to their own notebooks, because a Kaggle session only
# ever loads one model, and the cells are chunked so that one notebook fits one
# session (a Kaggle re-run starts clean, so nothing can be resumed).
all_mine = [c for c in plan.get("cells_preview", []) if c.get("model") == SERVED_MODEL]
mine = all_mine[CHUNK_START : CHUNK_START + CELLS]

print(f"arm {ARM} grid: {plan['cells']} cells total, {len(all_mine)} for {SERVED_MODEL}")
print(
    f"this notebook runs chunk {CHUNK_START // CELLS + 1}: "
    f"cells {CHUNK_START}..{CHUNK_START + len(mine) - 1}"
)
print()
for cell in mine:
    print(
        f"  {cell['name']:<30} preset={cell['preset']!s:<18} "
        f"seed={cell['seed_index']} prompt={cell['prompt_variant']!s:<18} "
        f"gov={cell['governance']!s:<10} beta={cell['beta']}"
    )

if len(mine) != CELLS:
    raise SystemExit(
        f"Expected {CELLS} cells in this chunk but the grid returned {len(mine)}. "
        "The grid manifest changed; regenerate the notebooks."
    )
if any(c.get("preset") is None or c.get("seed_index") is None for c in mine):
    raise SystemExit("A cell is missing preset/seed_index, so it cannot be run.")

if plan.get("parity_violations"):
    raise SystemExit(
        "PARITY VIOLATIONS - refusing to spend GPU-hours:\n"
        + json.dumps(plan["parity_violations"], indent=2)
    )
print("\nparity check passed.")

In [ ]:
# ============================================================================
# Preflight
# ============================================================================
# Each check corresponds to a way a run can silently become the wrong experiment:
#   governance_matches_cell          the cell runs at a governance level it did not ask for
#   chat_template_fixed_for_parity  the same prompt is a different string per model
#   seed_vector_complete            "four seeds" may actually be one draw
#   shocks_decoupled_from_democracy shock response == rule-change response
#   served_model_name_matches       the whole arm gets relabelled
#
# The cell is passed to `doctor` explicitly. Without it, doctor would describe the
# module defaults, which carry no seed vector and a server-side chat template, and
# would report two failures that say nothing about the run that is about to start.
first = mine[0]
_code, findings = elicit(
    "doctor",
    "--model",
    SERVED_MODEL,
    "--preset",
    str(first["preset"]),
    "--seed-index",
    str(first["seed_index"]),
)
print(json.dumps(findings, indent=2))

if findings.get("ok"):
    print(f"\npreflight passed for {first['name']}.")
else:
    failed = [f["check"] for f in findings["findings"] if not f["ok"]]
    print(f"\nfailing checks: {', '.join(failed)}")
    if failed == ["llm_reachable"]:
        print("The server may still be loading. Re-run this cell in a minute or two.")
    else:
        raise SystemExit(
            "Preflight failed on something other than server reachability. "
            "Stop here - running anyway would spend quota on the wrong experiment."
        )

In [ ]:
# ============================================================================
# Run this notebook's cells
# ============================================================================
# `mine` is this notebook's slice of the arm, in grid order. `cell["preset"]` and
# `cell["seed_index"]` are the coordinates `elicit run` needs; the cell *name* is
# not parseable back into them (arm_a_p0_qwen_s1 is preset a_qwen_p0, seed 1).
started = time.time()
completed, failed = [], []

for index, cell in enumerate(mine, start=1):
    elapsed = time.time() - started
    projected = elapsed + (len(mine) - index + 1) * PER_CELL_HOURS * 3600
    print("\n" + "=" * 72)
    print(
        f"[{index}/{len(mine)}] {cell['name']}  preset={cell['preset']} seed={cell['seed_index']}"
    )
    print(
        f"elapsed {elapsed / 3600:.2f} h | projected finish "
        f"{(projected + SETUP_ALLOWANCE_S) / 3600:.1f} h into the session"
    )
    print("=" * 72, flush=True)

    cell_started = time.time()
    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "cli",
            "run",
            "--preset",
            str(cell["preset"]),
            "--seed-index",
            str(cell["seed_index"]),
        ],
        cwd=str(REPO_DIR),
        env={**os.environ, "PYTHONPATH": str(REPO_DIR / "src")},
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        failed.append({"cell": cell["name"], "error": result.stderr[-2000:]})
        print(f"FAILED {cell['name']}:\n{result.stderr[-2000:]}")
        continue

    completed.append(cell["name"])
    took = (time.time() - cell_started) / 3600
    print(f"DONE {cell['name']} in {took:.2f} h")

elapsed_h = (time.time() - started) / 3600
print("\n" + "=" * 72)
print(
    json.dumps(
        {
            "arm": ARM,
            "served_model": SERVED_MODEL,
            "cells_planned": len(mine),
            "completed": completed,
            "failed": failed,
            "simulated_hours": round(elapsed_h, 2),
            "session_hours_including_setup": round(elapsed_h + SETUP_ALLOWANCE_S / 3600, 2),
            "kaggle_cap_hours": 12.0,
        },
        indent=2,
    )
)

if not failed and len(completed) == len(mine):
    print("\n" + "=" * 72)
    print(f"ALL {len(mine)} CELLS COMPLETE.")
    print("Download the archive below, then run the next numbered notebook.")
else:
    print("\n" + "=" * 72)
    print(f"{len(completed)} of {len(mine)} cells succeeded; {len(failed)} failed.")
    print("A Kaggle session cannot be resumed, so re-running repeats finished cells.")
    print("Read the failure above before re-running - a failed cell will fail again.")

In [ ]:
# ============================================================================
# Package the results - this is the deliverable
# ============================================================================
# Named with the arm, model and part so several sessions' outputs stay
# distinguishable when you extract them all into results/.
slug = (ARM + "_" + SERVED_MODEL + "part1").replace(".", "").replace("-", "_")
slug = slug.replace("__part", "part").rstrip("_")
archive = Path("/kaggle/working/elicit_results_" + slug + ".tar.gz")
if archive.exists():
    archive.unlink()

!cd {REPO_DIR} && tar -czf {archive} results/

print("archive:")
!ls -lh {archive}
print("\nresult files produced:")
!ls -1 {REPO_DIR}/results/simulation_*.json 2>/dev/null | wc -l

from IPython.display import FileLink  # noqa: E402

display(FileLink(str(archive)))
print("\nDownload elicit_results_" + slug + ".tar.gz above, then run the next notebook.")